In [1]:
import sys
from pathlib import Path

# go from /notebooks → project root
project_root = Path().resolve().parent
sys.path.append(str(project_root))

print(project_root)

D:\Big Boi Files\projects\langchain_practice\Microplastics-Research-Assistant-RAG-System-


In [2]:
# temporary bug workaround. see link for more permament solution. 
# https://github.com/vibrantlabsai/ragas/issues/2753#issuecomment-4563590504
import types
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [3]:
from rag.pipeline import run_rag
from rag.retriever import get_retriever
from rag.ingest import *

# load PDFs
docs = load_documents()

# chunk PDFs
split_docs = split_documents(docs)

# setup system
vector_store = build_vectorstore(split_docs)
retriever = get_retriever(vector_store)

# run experiments
samples = [
    run_rag("What are microplastics doing to human health?", retriever),
    run_rag("How do microplastics enter the ocean?", retriever),
]

c:\Users\dorky\anaconda3\envs\ragLLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pprint import pprint
pprint(samples)

[{'answer': 'Microplastics have been associated with several adverse effects '
            'on human health, although the research is still limited. In vitro '
            'studies have shown that microplastics can increase the frequency '
            'of micronucleation, nucleoplasm bridge formation, and nuclear bud '
            'formation in human blood lymphocytes, which may be linked to '
            'disorders such as infertility, diabetes, obesity, and '
            'cardiovascular disease. Additionally, while there is insufficient '
            'information to draw firm conclusions about the toxicity of '
            'microplastics, some animal studies have reported impacts like '
            'liver inflammation. The potential for chemical toxicity due to '
            'leaching of plastic-associated chemicals is also a concern. '
            'Overall, the human health effects of microplastics remain largely '
            'unknown and require further investigation.',
  'context

In [ ]:
import json

with open("../data/eval/ragas_samples.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)

In [6]:
from src.eval import build_ragas_dataset
dataset = build_ragas_dataset(samples)

In [7]:
print(dataset)

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response'], len=2)


### Below is experimental and needs modularized

In [ ]:
!python ../eval/run_eval.py

In [15]:
# read in results
import pandas as pd
df = pd.read_json("../data/results/eval_results.json")
df.head()

,faithfulness,answer_relevancy,context_relevance,scope_representation
0,1,0,1,3
1,1,1,1,4


In [16]:
# now average the scores for each metric
df_avg_scores = df.mean()

print("Evaluation Results:")
print("-------------------")
for metric, score in df_avg_scores.items():
    print(f"{metric}: {score:.4f}")

Evaluation Results:
-------------------
faithfulness: 1.0000
answer_relevancy: 0.5000
context_relevance: 1.0000
scope_representation: 3.5000
